# RAG mit Source Code: Text-Treffer gegen AST-Kontext

**Fortsetzung von `03-source-code-ast.ipynb`** – getrenntes Notebook zur LLM-Übung.

**Lernfrage:** Kann ein LLM einen Programmablauf korrekt erklären, wenn wir ihm nur die vier besten Texttreffer geben? Was ändert sich, wenn wir die mittels AST gefundenen möglichen Aufrufziele ergänzen?

**Was wir vergleichen**
1. **Einfaches Text-RAG:** eine Frage + vier Texttreffer.
2. **AST-erweitertes RAG:** dieselbe Frage + strukturell gefundene Funktionsdefinitionen.

Beide Varianten verwenden **`gemma3:4b`** und dieselben Antwortregeln. Die Codefragmente stammen aus den Autoshop-MS-Repositories, die Notebook 03 untersucht hat.

**Wichtig:** Eine Top-k-Suche beweist keine Vollständigkeit; auch AST-Kandidaten sind ohne Typ-/Laufzeitanalyse nicht immer eindeutig.

## 0. Vorbereitung: Ergebnisse aus Notebook 03 lesen

Zuerst `03-source-code-ast.ipynb` bis zum Schluss ausführen. Das Notebook speichert seine Retrieval-Ergebnisse in `artifacts/source-code-python-ast.json`.

Notebook 13 lädt diese Datei und benötigt **keine Parser-Pakete, keinen GitLab-Zugriff und keine neue Modellinstallation**. Ollama und `gemma3:4b` müssen für die anschliessende Abfrage bereits vorhanden sein.

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import json, os

ARTIFACT = Path("artifacts/source-code-python-ast.json")
if not ARTIFACT.is_file():
    raise FileNotFoundError("Zuerst 03-source-code-ast.ipynb ausführen. Erwartete Datei: " + str(ARTIFACT.resolve()))
experiment = json.loads(ARTIFACT.read_text(encoding="utf-8"))
if experiment.get("version") != 1:
    raise ValueError("Unbekanntes Exportformat. Notebook 03 nochmals ausführen.")

question = experiment["selected_question"]
text_context = [x["chunk"] for x in experiment["text_hits"]]
ast_context = experiment["ast_call_context"]
print("FRAGE:", question)
print("\nText-Retrieval:")
for c in text_context:
    print(f'  {c["repo"]}/{c["path"]}:{c["start_line"]}  {c["symbol"]}')
print("\nAST-Kontext:")
for c in ast_context:
    print(f'  {c["repo"]}/{c["path"]}:{c["start_line"]}  {c["symbol"]}')

## 1. Identische Frage, unterschiedlicher Kontext

**Was machen wir?** Wir fragen das lokale Sprachmodell zweimal. Zwischen den Durchläufen ändern wir **nur die ausgewählten Code-Stellen**.

Die Antwort soll zu jeder Aussage die Fundstelle nennen und unsichere Aufrufbeziehungen nicht als Tatsache darstellen. Falls der Kontext nicht genügt, soll das Modell die Grenze ausdrücklich nennen.

Beobachte: Liefert der AST-Kontext mehr relevante Definitionen? Entstehen trotzdem unbelegte Schlussfolgerungen?

In [ ]:
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434").rstrip("/")
MODEL = "gemma3:4b"
MAX_CONTEXT_CHARS = 7000


def make_context(chunks):
    parts, length = [], 0
    for chunk in chunks:
        reference = f'[{chunk["repo"]}/{chunk["path"]}:{chunk["start_line"]} | {chunk["symbol"]}]'
        remaining = MAX_CONTEXT_CHARS - length - len(reference) - 5
        if remaining < 100:
            break
        excerpt = chunk["content"][:remaining]
        parts.append(reference + "\n" + excerpt)
        length += len(reference) + len(excerpt) + 5
    return "\n\n---\n\n".join(parts)


def ask_ollama(query, chunks):
    instruction = (
        "Beantworte die Frage ausschliesslich anhand des übergebenen Python-Source-Codes. "
        "Nenne für jede wesentliche Aussage Repository, Dateipfad und Zeile. "
        "Kennzeichne mögliche Aufrufe als Kandidaten und unterscheide sie von gesicherten Fakten. "
        "Wenn wichtige Dateien oder Laufzeitinformationen fehlen, benenne diese Lücken.\n\n"
        "FRAGE: " + query + "\n\nSOURCE CODE:\n" + make_context(chunks)
    )
    data = json.dumps({"model": MODEL, "prompt": instruction, "stream": False,
                       "options": {"temperature": 0}}).encode("utf-8")
    request = Request(OLLAMA_URL + "/api/generate", data=data,
                      headers={"Content-Type": "application/json"}, method="POST")
    with urlopen(request, timeout=180) as response:
        return json.load(response).get("response", "<leere Antwort>")


try:
    with urlopen(OLLAMA_URL + "/api/tags", timeout=5) as response:
        available = [m["name"] for m in json.load(response).get("models", [])]
    if MODEL not in available:
        print(f"{MODEL} ist nicht installiert. Vorhandene Modelle: {available}")
    else:
        print("=== A: Einfaches Text-RAG ===")
        print(ask_ollama(question, text_context))
        print("\n=== B: AST-erweitertes RAG ===")
        print(ask_ollama(question, ast_context))
except (HTTPError, URLError, TimeoutError, OSError) as exc:
    print("Ollama-Abfrage nicht möglich:", exc)

## 2. Ergebnisse beurteilen

**Kontrollfragen zum Vergleich**

- Welche Dateien aus dem AST-Kontext fehlten bei den vier Texttreffern?
- Welche Aussagen beider Antworten sind anhand der übergebenen Fundstellen nachprüfbar?
- Welche Aussagen über das Laufzeitverhalten bleiben trotz AST-Kontext unbelegt?

**Erkenntnis:** Retrieval bestimmt, welche Hinweise ein LLM überhaupt bekommt. AST verbessert das Auffinden und Zusammenführen von Code-Strukturen; eine sichere Ende-zu-Ende-Aussage braucht gegebenenfalls zusätzlich Typauflösung, Tests und Laufzeittraces.

Ein schlechteres Text-RAG-Ergebnis ist **kein genereller Beweis, dass RAG bei Code nicht funktioniert** – sondern ein konkreter Vergleich zweier Kontexte am ausgewählten Codebeispiel.